# Compute & RAM per architecture

Per-model MFLOPs, flash and arena read from each baked int8 tflite, plus the theoretical streaming line-buffer arena.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os

import pyrootutils

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator='pyproject.toml',
    pythonpath=True,
    dotenv=True,
)

## Models

In [ ]:
from building.benching.benching import ensure_bench
from building.models import EXTENDED_LIST

MODELS = EXTENDED_LIST
ensure_bench(MODELS)

## Baked compute & RAM

In [ ]:
import pandas as pd

from building.models.compute_cost import baked_rows


def fmt(x: float) -> str:
    if x == 0:
        return '0'
    a = abs(x)
    if a >= 100: return f'{x:.0f}'
    if a >= 10:  return f'{x:.1f}'
    if a >= 1:   return f'{x:.2f}'
    return f'{x:.3g}'


pd.set_option('display.float_format', fmt)

df = pd.DataFrame([r.model_dump() for r in baked_rows(MODELS)]).sort_values('mflops', ascending=False).reset_index(drop=True)
df